In [1]:
import numpy as np
import matplotlib.pyplot as plt
# import rosbags
from pathlib import Path
from ekf.class_ekf import EKF
from data.ford_data_adapter import FordDataAdapter

In [2]:
adapter = FordDataAdapter(Path("2017-10-26-V2-Log6.bag"))

[INFO]  Data folder 2017-10-26-V2-Log6 already exists. Not creating.


In [3]:
first_update = True
ekf = EKF()
estimations_x = []
estimations_y = []
ground_truth_x = []
ground_truth_y = []

In [ ]:
first_update = True

estimations_x = []
estimations_y = []
ground_truth_x = []
ground_truth_y = []

for i, (action, value, timestamp) in enumerate(adapter):
    if action == 'predict':
        ekf.predict(value)
    elif action == 'update':
        if first_update:
            ekf.update(value)
            first_update = False
        else:
            ekf.update(value)
        estimations_x.append(ekf.x[0])
        estimations_y.append(ekf.x[1])
    elif action == 'ground_truth':
        ground_truth_x.append(value[0])
        ground_truth_y.append(value[1])
    
    if i > 5000:
        break

start_idx = 0
for i in range(1, len(ground_truth_x)):
    if ground_truth_x[i] != ground_truth_x[0]:
        start_idx = i
        break


estimations_x = estimations_x[start_idx:]
estimations_y = estimations_y[start_idx:]
ground_truth_x = ground_truth_x[start_idx:]
ground_truth_y = ground_truth_y[start_idx:]

est_offset_x = estimations_x[0]
est_offset_y = estimations_y[0]
gt_offset_x = ground_truth_x[0]
gt_offset_y = ground_truth_y[0]
    
est_x_plot = [x - est_offset_x for x in estimations_x]
est_y_plot = [y - est_offset_y for y in estimations_y]
gt_x_plot = [x - gt_offset_x for x in ground_truth_x]
gt_y_plot = [y - gt_offset_y for y in ground_truth_y]
    
plt.figure(figsize=(10, 8))
plt.plot(gt_x_plot, gt_y_plot, 'g-', linewidth=5, label='Ground Truth')
plt.plot(est_x_plot, est_y_plot, 'b-', linewidth=2, label='EKF')
plt.xlabel('X (м)')
plt.ylabel('Y (м)')
plt.title('Сравнение EKF с эталонной траекторией')
plt.legend()
plt.grid(True)
plt.axis('equal')

plt.savefig(Path("results") / "comparison_2017-10-26-V2-Log6.png", dpi=150, bbox_inches='tight')
plt.close()